# 01 Discovery and ground truth (slice 1)

Goal: find the DNS tunnel in `dns.log`, confirm it in `case1.pcap`, label every row as `benign` or `tunnel`, and save `data/labels.csv`.

**Row id convention (shared with the group):** `row_id` is the 0-based position of the row among the data rows of `dns.log` (header lines starting with `#` skipped, file order kept).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

# Zeek names the columns itself: the line starting with #fields lists them in order
with open(RAW / "dns.log", encoding="latin-1") as f:
    COLS = next(line for line in f if line.startswith("#fields")).rstrip("\n").split("\t")[1:]

df = pd.read_csv(RAW / "dns.log", sep="\t", comment="#", names=COLS,
                 na_values=["-"], keep_default_na=False, encoding="latin-1")
df.index.name = "row_id"
df["dt"] = pd.to_datetime(df["ts"], unit="s")
df["qlen"] = df["query"].str.len()
print(len(COLS), "columns read from the #fields line")
print(df.shape, df["dt"].min(), "to", df["dt"].max())

## 1. First look: who talks, with what

In [ ]:
display(df["id.orig_h"].value_counts().to_frame("queries"))
display(df["qtype_name"].value_counts().to_frame("queries"))
display(df["rcode_name"].value_counts(dropna=False).to_frame("queries"))

## 2. Rank parent domains: which one does not look like the others?

In [ ]:
df["parent"] = df["query"].str.split(".").str[-2:].str.join(".")
parents = (df.groupby("parent")
             .agg(queries=("query", "size"), unique_names=("query", "nunique"),
                  mean_len=("qlen", "mean"), max_len=("qlen", "max"),
                  clients=("id.orig_h", "nunique"))
             .sort_values("unique_names", ascending=False))
parents.head(10)

`pirate.sea` has by far the most unique names and by far the longest names. Next: is it also one client and one record type?

In [ ]:
is_domain = df["query"].str.endswith(".pirate.sea")
is_null = df["qtype_name"] == "NULL"
is_host = df["id.orig_h"] == "10.0.2.30"
display(pd.crosstab([is_domain.rename("pirate.sea"), is_null.rename("qtype NULL")], is_host.rename("client 10.0.2.30")))
print("longest benign name:", df.loc[~is_domain, "qlen"].max(), "chars; longest tunnel name:", df.loc[is_domain, "qlen"].max())

## 3. Timeline of the tunnel

In [ ]:
t = df[is_domain]
print("first:", t["dt"].min(), " last:", t["dt"].max(), " rows:", len(t), " unique names:", t["query"].nunique())
print("rows without response (rcode empty):", t["rcode_name"].isna().sum())
display(t.groupby(t["dt"].dt.floor("5min")).size().to_frame("tunnel queries"))

fig, ax = plt.subplots(figsize=(10, 3))
per_min = df.groupby([df["dt"].dt.floor("1min"), is_domain.rename("tunnel")]).size().unstack(fill_value=0)
per_min.plot(ax=ax, kind="bar", stacked=True, width=1.0, legend=True)
ax.set_xticks(ax.get_xticks()[::10]); ax.set_ylabel("queries per minute"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Is the second burst a repeat of the first? Compare the names sent in each burst.
split = pd.Timestamp("2023-01-07 18:30:00")
first, second = set(t.loc[t["dt"] < split, "query"]), set(t.loc[t["dt"] >= split, "query"])
print("burst 1:", len(first), "names | burst 2:", len(second), "names | shared:", len(first & second))
t["query"].head(5).tolist()

## 4. Confirm in the PCAP

`dns.log` is Zeek's view. The PCAP is the ground truth for what was on the wire. We check that the packet-level count of DNS queries matches the log and that the `pirate.sea` queries are NULL queries from `10.0.2.30`.

In [ ]:
from scapy.all import rdpcap, DNS, DNSQR, IP
pkts = rdpcap(str(RAW / "case1.pcap"))
print("packets in pcap:", len(pkts))

rows = []
for p in pkts:
    if p.haslayer(DNS) and p.haslayer(IP):
        d = p[DNS]
        qname = d.qd.qname.decode("latin-1").rstrip(".") if d.qd else None
        rows.append((float(p.time), p[IP].src, p[IP].dst, d.qr, d.qd.qtype if d.qd else None, qname, len(p)))
pc = pd.DataFrame(rows, columns=["ts", "src", "dst", "is_response", "qtype", "qname", "frame_len"])
pc["dt"] = pd.to_datetime(pc["ts"], unit="s")
print("DNS packets:", len(pc), "| queries:", (pc["is_response"] == 0).sum(), "| responses:", (pc["is_response"] == 1).sum())
pc_t = pc[pc["qname"].fillna("").str.endswith(".pirate.sea")]
print("pirate.sea packets:", len(pc_t), "| qtype values:", pc_t["qtype"].unique(), "| sources of queries:", pc_t.loc[pc_t.is_response == 0, "src"].unique())
display(pc_t.groupby("is_response")["frame_len"].describe())

## 5. Label and save

**Rule:** `tunnel` = query name under `pirate.sea`. Everything else is `benign`. The rule agrees with the client (`10.0.2.30`) and the record type (NULL) criteria on every row, shown in section 2.

In [ ]:
labels = pd.DataFrame({"label": np.where(is_domain, "tunnel", "benign")}, index=df.index)
assert (is_domain == is_null).all() and (is_domain == is_host).all(), "criteria disagree, inspect before labelling"
labels.to_csv(ROOT / "data" / "labels.csv")
print(labels["label"].value_counts())
print("tunnel share: %.1f%%" % (100 * (labels["label"] == "tunnel").mean()))

## 6. Sanity checks on the saved file

In [ ]:
chk = pd.read_csv(ROOT / "data" / "labels.csv", index_col="row_id")
assert len(chk) == len(df) and chk.index.equals(df.index)
print("rows match dns.log:", len(chk))
print("random benign sample (read by eye):")
display(df.loc[chk.index[chk["label"] == "benign"], ["id.orig_h", "query", "qtype_name"]].sample(15, random_state=1))
print("benign rows that look odd (NXDOMAIN or long name):")
display(df.loc[(chk["label"] == "benign") & ((df["rcode_name"] == "NXDOMAIN") | (df["qlen"] > 50)),
               ["id.orig_h", "query", "qtype_name", "rcode_name"]])

## Summary: what we did and why

1. **Loaded `dns.log`** (3,645 DNS queries). Column names come from the log's own `#fields` line.
2. **Looked at who talks and with what.** Two clients on two separate networks, each with its own resolver. One client sends only NULL records.
3. **Ranked parent domains.** `pirate.sea` stands out on unique names (222), name length and record type. It does not stand out on query volume.
4. **Cross-checked the suspicion.** Domain, record type NULL and client `10.0.2.30` agree on every row, so one simple rule labels everything without conflicts.
5. **Plotted the timeline.** Two bursts of about a minute, 46 minutes apart (18:04 and 18:50 to 18:51). The second burst repeats the same 222 names as the first.
6. **Confirmed in the PCAP.** The log is complete (3,645 queries, 21 unanswered). The PCAP also holds the answer payload that Zeek cannot decode.
7. **Labelled and saved** `data/labels.csv`: 444 tunnel, 3,201 benign (12.2% tunnel). The rule is "the name ends in `pirate.sea`".
8. **Sanity-checked the labels.** Row counts match, a random benign sample looks like normal browsing, and the odd-looking benign rows (long cloud names, NXDOMAIN) are not tunnel traffic. They are the likely false positives later.